In [2]:
import csv
import random
from datetime import datetime, timedelta
import sqlite3
import pandas as pd

In [3]:
# Data Generation

def generate_cellular_data(num_rows):

    with open('cellular_data.csv', 'w', newline='') as file:

        writer = csv.writer(file)

        writer.writerow(['customer_id', 'customer_name', 'plan_id', 'plan_name', 'monthly_fee', 'data_usage_gb', 'contract_start_date'])

        

        plans = [

            (1, 'Basic', 29.99),

            (2, 'Standard', 49.99),

            (3, 'Premium', 79.99),

            (4, 'Unlimited', 99.99)

        ]

        

        for i in range(num_rows):

            customer_id = random.randint(1000, 9999)

            customer_name = f"Customer {customer_id}"

            plan = random.choice(plans)

            plan_id, plan_name, monthly_fee = plan

            data_usage_gb = round(random.uniform(0, 50), 2)

            contract_start_date = datetime.now() - timedelta(days=random.randint(0, 730))

            

            writer.writerow([customer_id, customer_name, plan_id, plan_name, monthly_fee, data_usage_gb, contract_start_date])


In [4]:
# Data Storage and Ingestion

def create_tables(conn):

    conn.execute('''CREATE TABLE IF NOT EXISTS customers

                    (customer_id INTEGER PRIMARY KEY,

                     customer_name TEXT)''')

    

    conn.execute('''CREATE TABLE IF NOT EXISTS plans

                    (plan_id INTEGER PRIMARY KEY,

                     plan_name TEXT,

                     monthly_fee REAL)''')

    

    conn.execute('''CREATE TABLE IF NOT EXISTS subscriptions

                    (subscription_id INTEGER PRIMARY KEY AUTOINCREMENT,

                     customer_id INTEGER,

                     plan_id INTEGER,

                     data_usage_gb REAL,

                     contract_start_date DATE,

                     FOREIGN KEY (customer_id) REFERENCES customers (customer_id),

                     FOREIGN KEY (plan_id) REFERENCES plans (plan_id))''')

 

def load_data(conn):

    cellular_data = pd.read_csv('cellular_data.csv')

    

    customers = cellular_data[['customer_id', 'customer_name']].drop_duplicates()

    plans = cellular_data[['plan_id', 'plan_name', 'monthly_fee']].drop_duplicates()

    subscriptions = cellular_data[['customer_id', 'plan_id', 'data_usage_gb', 'contract_start_date']]

    

    customers.to_sql('customers', conn, if_exists='replace', index=False)

    plans.to_sql('plans', conn, if_exists='replace', index=False)

    subscriptions.to_sql('subscriptions', conn, if_exists='append', index=False)

In [5]:
# Data Transformation

def transform_data(conn):

    # Calculate total revenue for each plan

    query = '''

        CREATE VIEW IF NOT EXISTS plan_revenue AS

        SELECT p.plan_name, SUM(p.monthly_fee) AS total_revenue

        FROM subscriptions s

        JOIN plans p ON s.plan_id = p.plan_id

        GROUP BY p.plan_name

    '''

    conn.execute(query)

    

    # Find top 5 customers by data usage

    query = '''

        CREATE VIEW IF NOT EXISTS top_data_users AS

        SELECT c.customer_name, SUM(s.data_usage_gb) AS total_data_usage

        FROM subscriptions s

        JOIN customers c ON s.customer_id = c.customer_id

        GROUP BY c.customer_name

        ORDER BY total_data_usage DESC

        LIMIT 5

    '''

    conn.execute(query)

    

    # Determine average data usage per plan

    query = '''

        CREATE VIEW IF NOT EXISTS avg_plan_data_usage AS

        SELECT p.plan_name, AVG(s.data_usage_gb) AS avg_data_usage

        FROM subscriptions s

        JOIN plans p ON s.plan_id = p.plan_id

        GROUP BY p.plan_name

    '''

    conn.execute(query)

    

    # Identify the most popular plan

    query = '''

        CREATE VIEW IF NOT EXISTS popular_plan AS

        SELECT p.plan_name, COUNT(*) AS subscription_count

        FROM subscriptions s

        JOIN plans p ON s.plan_id = p.plan_id

        GROUP BY p.plan_name

        ORDER BY subscription_count DESC

        LIMIT 1

    '''

    conn.execute(query)

In [6]:
# Main execution

if __name__ == "__main__":

    generate_cellular_data(1000)  # Generate 1000 rows of data

    

    conn = sqlite3.connect('cellular_company.db')

    create_tables(conn)

    load_data(conn)

    transform_data(conn)

    

    conn.close()

    

    print("Data generation, storage, and transformation complete.")

Data generation, storage, and transformation complete.
